# FoodCal: เทรน Food-101 และ THFOOD-50 ใหม่แบบแยกโมเดล

Notebook นี้สร้างโมเดลใหม่จาก ImageNet-pretrained EfficientNet-B0 ทั้งสองชุด โดยไม่แก้ไฟล์หรือ checkpoint เดิม

- Food-101: stratified 80/10/10, seed 42 เพื่อให้ตรงกับแผนและ split เดิม
- THFOOD-50: รวมภาพจากโฟลเดอร์ train/val แล้วแบ่งใหม่แบบ stratified 80/10/10, seed 42 ตามแนวทางเดิม
- เลือก checkpoint จาก validation เท่านั้น; ประเมิน test หลังเทรนเสร็จ
- Food-101 ใช้สูตรเดิมที่เคยได้ 85.02%: 10 epochs, AdamW lr=5e-4, augmentation เดิม และเลือก checkpoint จาก validation
- บันทึกผลรอบใหม่แยกโฟลเดอร์ ไม่ทับ checkpoint รอบก่อน
- ผลลัพธ์ ONNX, class order และ metrics แยกโฟลเดอร์สำหรับแต่ละโมเดล

> ต้องเปิด GPU ก่อนรัน training cell ถ้า Colab ไม่มี GPU ให้หยุด อย่าปล่อยให้เทรนบน CPU
> THFOOD-50 ระบุเงื่อนไข non-commercial research/educational ตามแหล่งชุดข้อมูล โปรดใช้เฉพาะตาม license


In [ ]:
# ติดตั้งแพ็กเกจที่ต้องใช้ (ไม่อัปเกรด PyTorch/torchvision ของ Colab)
!pip -q install timm scikit-learn onnx onnxruntime gdown

import os, json, random, shutil, time, copy, math
from pathlib import Path
from collections import Counter
import numpy as np
from PIL import Image
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms
import timm
from sklearn.metrics import accuracy_score, f1_score, balanced_accuracy_score, confusion_matrix, classification_report

SEED = 42
def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
seed_everything()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch:', torch.__version__, '| timm:', timm.__version__, '| device:', device)
if device.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
    import subprocess
    print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
else:
    print('ยังไม่พบ GPU: Runtime > Change runtime type > T4 GPU แล้วรัน cell นี้ใหม่ก่อนเทรน')


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive')
OUTPUT_ROOT = DRIVE_ROOT / 'FoodCal_Retraining_2026'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print('ไฟล์ผลลัพธ์จะเก็บที่:', OUTPUT_ROOT)


## 1) เตรียม Food-101

สร้าง split ใหม่จากภาพทั้งหมดตาม 80/10/10 ที่เสนอไว้ โดยสุ่มภายในแต่ละคลาสด้วย seed 42 และสร้าง EfficientNet ใหม่จาก pretrained weights เท่านั้น ไม่โหลด checkpoint เก่ามาเทรนต่อ


In [ ]:
# Download/extract Food-101 (ประมาณ 5 GB)
food_ds = datasets.Food101(root='/content/data', split='train', download=True)
FOOD_ROOT = Path('/content/data/food-101')
FOOD_IMAGE_ROOT = FOOD_ROOT / 'images'
food_classes = (FOOD_ROOT / 'meta' / 'classes.txt').read_text().split()
assert len(food_classes) == 101
assert set(food_classes) == {p.name for p in FOOD_IMAGE_ROOT.iterdir() if p.is_dir()}

def split_food101(seed=SEED):
    random.seed(seed)  # สุ่มต่อเนื่องตามลำดับ classes.txt เหมือน split เดิม
    out = {'train': [], 'val': [], 'test': []}
    for label_id, cls in enumerate(food_classes):
        files = sorted((FOOD_IMAGE_ROOT / cls).glob('*.jpg'))
        assert len(files) == 1000, (cls, len(files))
        random.shuffle(files)
        out['train'] += [(str(p), label_id) for p in files[:800]]
        out['val']   += [(str(p), label_id) for p in files[800:900]]
        out['test']  += [(str(p), label_id) for p in files[900:]]
    return out

food_split = split_food101()
print({k: len(v) for k, v in food_split.items()})
assert [len(food_split[k]) for k in ('train','val','test')] == [80800, 10100, 10100]
(OUTPUT_ROOT / 'split_food101_80_10_10.json').write_text(json.dumps({k: [(Path(p).relative_to(FOOD_ROOT).as_posix(), y) for p,y in v] for k,v in food_split.items()}, indent=2))


## 2) เตรียม THFOOD-50

Notebook จะดาวน์โหลด zip จาก Google Drive ลิงก์สาธารณะของ repository ผู้จัดทำให้อัตโนมัติ ถ้ามีไฟล์ `THFOOD50-v1.zip` อยู่ใน `MyDrive` อยู่แล้วก็ใช้ไฟล์นั้นแทนได้


In [ ]:
import zipfile, gdown
THFOOD50_FILE_ID = '1CuNO2e77ZTk7mDfv3XujYXuUwiMwlUQI'
THAI_DRIVE_ZIP = DRIVE_ROOT / 'THFOOD50-v1.zip'
THAI_LOCAL_ZIP = Path('/content/THFOOD50-v1.zip')
THAI_EXTRACT = Path('/content/thfood50')

if THAI_DRIVE_ZIP.exists():
    THAI_ZIP = THAI_DRIVE_ZIP
    print('ใช้ไฟล์ที่มีอยู่แล้วใน My Drive:', THAI_ZIP)
else:
    THAI_ZIP = THAI_LOCAL_ZIP
    if not THAI_ZIP.exists():
        print('กำลังดาวน์โหลด THFOOD-50 จาก Google Drive ของผู้จัดทำ...')
        downloaded = gdown.download(id=THFOOD50_FILE_ID, output=str(THAI_ZIP), quiet=False)
        if downloaded is None or not THAI_ZIP.exists():
            raise RuntimeError('ดาวน์โหลดอัตโนมัติไม่สำเร็จ หาก Google Drive จำกัดโควตา ให้วาง THFOOD50-v1.zip ใน MyDrive แล้วรัน cell นี้ใหม่')

if not zipfile.is_zipfile(THAI_ZIP):
    raise RuntimeError('ไฟล์ที่ได้ไม่ใช่ ZIP ที่ถูกต้อง หากดาวน์โหลดจาก Drive ไม่ผ่าน ให้อัปโหลด THFOOD50-v1.zip ไป MyDrive แล้วรันใหม่')
if not THAI_EXTRACT.exists() or not any(THAI_EXTRACT.rglob('readme.txt')):
    THAI_EXTRACT.mkdir(parents=True, exist_ok=True)
    shutil.unpack_archive(str(THAI_ZIP), str(THAI_EXTRACT))
train_dirs = [p for p in THAI_EXTRACT.rglob('train') if (p.parent / 'val').is_dir()]
assert len(train_dirs) == 1, f'หาโฟลเดอร์ train/val ไม่เจอหรือพบหลายชุด: {train_dirs}'
THAI_ROOT = train_dirs[0].parent
THAI_TRAIN_DIR, THAI_VAL_DIR = THAI_ROOT / 'train', THAI_ROOT / 'val'
thai_classes = sorted(p.name for p in THAI_TRAIN_DIR.iterdir() if p.is_dir())
assert len(thai_classes) == 50, f'คาดหวัง 50 คลาส พบ {len(thai_classes)}'
assert set(thai_classes) == {p.name for p in THAI_VAL_DIR.iterdir() if p.is_dir()}

# ตรวจความสมบูรณ์ของภาพก่อนแบ่งข้อมูล เพื่อกันไฟล์เสียทำให้ DataLoader worker ล้มระหว่างเทรน
THAI_IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".webp"}
THAI_IMAGE_PATHS = sorted(
    p for cls in thai_classes
    for folder in (THAI_TRAIN_DIR / cls, THAI_VAL_DIR / cls)
    for p in folder.rglob("*")
    if p.is_file() and p.suffix.lower() in THAI_IMAGE_SUFFIXES
)
THAI_VALID_IMAGES = set()
THAI_BAD_IMAGES = []
for image_path in THAI_IMAGE_PATHS:
    try:
        with Image.open(image_path) as image:
            image.verify()
        with Image.open(image_path) as image:
            image.load()
        THAI_VALID_IMAGES.add(str(image_path.resolve()))
    except (OSError, ValueError, SyntaxError, Image.DecompressionBombError) as error:
        THAI_BAD_IMAGES.append((str(image_path), str(error)))
print(f"ตรวจภาพ {len(THAI_IMAGE_PATHS)} ไฟล์: ใช้ได้ {len(THAI_VALID_IMAGES)}, ข้ามไฟล์เสีย {len(THAI_BAD_IMAGES)}")
if THAI_BAD_IMAGES:
    print("ตัวอย่างไฟล์ที่ข้าม:")
    for image_path, error in THAI_BAD_IMAGES[:20]:
        print(" -", image_path, "|", error)

def split_thai(seed=SEED):
    rng = random.Random(seed)
    out = {'train': [], 'val': [], 'test': []}
    counts = {}
    for label_id, cls in enumerate(thai_classes):
        files = []
        for folder in (THAI_TRAIN_DIR / cls, THAI_VAL_DIR / cls):
            files += [p for p in folder.rglob('*') if p.is_file() and p.suffix.lower() in {'.jpg','.jpeg','.png','.webp'}]
        files = sorted(p for p in files if str(p.resolve()) in THAI_VALID_IMAGES)
        rng.shuffle(files)
        n = len(files); n_train = int(n * 0.8); n_val = int(n * 0.1)
        assert n_train > 0 and n_val > 0 and n - n_train - n_val > 0, (cls, n)
        out['train'] += [(str(p), label_id) for p in files[:n_train]]
        out['val']   += [(str(p), label_id) for p in files[n_train:n_train+n_val]]
        out['test']  += [(str(p), label_id) for p in files[n_train+n_val:]]
        counts[cls] = n
    return out, counts

thai_split, thai_counts = split_thai()
assert sum(thai_counts.values()) == len(THAI_VALID_IMAGES), f'จำนวนรูปที่แบ่งไม่ตรงกับภาพที่ผ่านการตรวจ: {sum(thai_counts.values())} != {len(THAI_VALID_IMAGES)}'
print('รวมรูป:', sum(thai_counts.values()), '| คลาส:', len(thai_classes))
print({k: len(v) for k,v in thai_split.items()})
print('คลาสน้อยสุด/มากสุด:', min(thai_counts.items(), key=lambda x:x[1]), max(thai_counts.items(), key=lambda x:x[1]))
(OUTPUT_ROOT / 'split_thfood50_80_10_10.json').write_text(json.dumps({k: [(str(Path(p).relative_to(THAI_ROOT)), y) for p,y in v] for k,v in thai_split.items()}, indent=2))


## 3) Data pipeline และ training function

สำหรับ Food-101 ให้ทำซ้ำสูตรที่เคยได้ Top-1 85.02% เพื่อเทียบผลอย่างเป็นธรรม: train augmentation แบบเดิม, AdamW lr=5e-4 สำหรับทุกพารามิเตอร์, label smoothing 0.1, OneCycleLR และ 10 epochs. ใช้ validation เลือก checkpoint; ประเมิน test หลังจบเท่านั้น.


In [ ]:
IMAGE_SIZE = 224
MAX_EPOCHS = 10
PATIENCE = MAX_EPOCHS + 1  # ปิด early stop เพื่อให้เทียบสูตรเดิมที่ฝึกครบ 10 epochs
BATCH_SIZE = 64
NUM_WORKERS = 2

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(0.2, 0.2, 0.2),
    transforms.ToTensor(),
    transforms.Normalize((0.485,0.456,0.406),(0.229,0.224,0.225)),
])
eval_tf = transforms.Compose([
    transforms.Resize(256), transforms.CenterCrop(IMAGE_SIZE), transforms.ToTensor(),
    transforms.Normalize((0.485,0.456,0.406),(0.229,0.224,0.225)),
])

class FoodImages(Dataset):
    def __init__(self, items, tf): self.items, self.tf = items, tf
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        path, label = self.items[i]
        with Image.open(path) as im: image = im.convert('RGB')
        return self.tf(image), label

def make_loaders(split):
    kwargs = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=(NUM_WORKERS > 0))
    return (DataLoader(FoodImages(split['train'], train_tf), shuffle=True, **kwargs),
            DataLoader(FoodImages(split['val'], eval_tf), shuffle=False, **kwargs),
            DataLoader(FoodImages(split['test'], eval_tf), shuffle=False, **kwargs))

def score_model(model, loader, n_classes):
    model.eval(); ys=[]; preds=[]; top3=[]
    with torch.inference_mode():
        for x,y in loader:
            logits = model(x.to(device, non_blocking=True))
            ys.extend(y.numpy().tolist())
            preds.extend(logits.argmax(1).cpu().numpy().tolist())
            top3.extend(logits.topk(min(3,n_classes), dim=1).indices.cpu().numpy().tolist())
    y=np.asarray(ys); p=np.asarray(preds); t=np.asarray(top3)
    return {
        'top1_accuracy': float(accuracy_score(y,p)),
        'top3_accuracy': float(np.mean([y[i] in t[i] for i in range(len(y))])),
        'macro_f1': float(f1_score(y,p,average='macro',zero_division=0)),
        'balanced_accuracy': float(balanced_accuracy_score(y,p)),
        'y_true': y, 'y_pred': p,
    }

def build_model(n_classes):
    model = timm.create_model('efficientnet_b0', pretrained=True, num_classes=n_classes).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=1e-2)
    return model, optimizer

def train_model(name, classes, split, selection_metric, out_dir):
    if device.type != 'cuda': raise RuntimeError('หยุด: ต้องเปิด GPU ก่อนเริ่ม training')
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    train_dl, val_dl, test_dl = make_loaders(split)
    model, optimizer = build_model(len(classes))
    scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=5e-4, epochs=MAX_EPOCHS, steps_per_epoch=len(train_dl))
    scaler = torch.amp.GradScaler('cuda')
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    history=[]; best=-1.0; stale=0; start_epoch=0
    last_path=out_dir / 'last_checkpoint.pt'; best_path=out_dir / 'best_weights.pt'
    if last_path.exists():
        ckpt=torch.load(last_path, map_location='cpu')
        model.load_state_dict(ckpt['model']); optimizer.load_state_dict(ckpt['optimizer'])
        scheduler.load_state_dict(ckpt['scheduler']); scaler.load_state_dict(ckpt['scaler'])
        history=ckpt['history']; best=ckpt['best']; stale=ckpt['stale']; start_epoch=ckpt['epoch']+1
        print(f'Resume {name}: ต่อจาก epoch {start_epoch+1}')
    elif best_path.exists():
        print(f'พบ best checkpoint ของ {name}; ข้ามการเทรนซ้ำและโหลด best weights')
        model.load_state_dict(torch.load(best_path, map_location='cpu'))
        return model, history, test_dl
    if stale >= PATIENCE:
        print('checkpoint หยุดด้วย early stopping แล้ว; ใช้ best weights')
        model.load_state_dict(torch.load(best_path, map_location='cpu'))
        return model, history, test_dl
    for epoch in range(start_epoch, MAX_EPOCHS):
        model.train(); loss_sum=0.0; seen=0
        for x,y in train_dl:
            x=x.to(device, non_blocking=True); y=y.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type='cuda', dtype=torch.float16):
                logits=model(x); loss=criterion(logits,y)
            scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update(); scheduler.step()
            loss_sum += loss.item()*len(y); seen += len(y)
        val=score_model(model,val_dl,len(classes))
        metric=val['top1_accuracy'] if selection_metric=='top1_accuracy' else val['macro_f1']
        row={'epoch':epoch+1,'train_loss':loss_sum/seen,'val_top1':val['top1_accuracy'],'val_top3':val['top3_accuracy'],'val_macro_f1':val['macro_f1']}
        history.append(row)
        improved=metric > best
        if improved:
            best=metric; stale=0
            torch.save(model.state_dict(),best_path)
        else: stale += 1
        torch.save({'epoch':epoch,'model':model.state_dict(),'optimizer':optimizer.state_dict(),'scheduler':scheduler.state_dict(),'scaler':scaler.state_dict(),'history':history,'best':best,'stale':stale},last_path)
        print(f"{name} | epoch {epoch+1:02d}/{MAX_EPOCHS} | loss {row['train_loss']:.4f} | val top1 {row['val_top1']:.4f} | val macro-F1 {row['val_macro_f1']:.4f} | best={improved}")
        if stale >= PATIENCE:
            print(f'Early stopping after {epoch+1} epochs'); break
    model.load_state_dict(torch.load(best_path,map_location='cpu')); model.to(device)
    (out_dir/'history.json').write_text(json.dumps(history,indent=2))
    return model, history, test_dl

def final_evaluate_and_export(name, slug, model, classes, test_dl, out_dir):
    out_dir=Path(out_dir); result=score_model(model,test_dl,len(classes))
    y,p=result.pop('y_true'),result.pop('y_pred')
    result['test_images']=int(len(y)); result['class_count']=len(classes); result['model']='EfficientNet-B0'; result['split']='stratified 80/10/10, seed 42'
    report=classification_report(y,p,target_names=classes,output_dict=True,zero_division=0)
    (out_dir/'test_metrics.json').write_text(json.dumps(result,indent=2))
    (out_dir/'per_class_report.json').write_text(json.dumps(report,indent=2))
    np.savetxt(out_dir/'confusion_matrix.csv',confusion_matrix(y,p),delimiter=',',fmt='%d')
    (out_dir/f'classes_{slug}.txt').write_text('\n'.join(classes)+'\n')
    (out_dir/f'classes_{slug}.json').write_text(json.dumps(classes,ensure_ascii=False,indent=2))
    model.eval().cpu()
    onnx_path=out_dir/f'{slug}_effnetb0.onnx'
    dummy=torch.randn(1,3,IMAGE_SIZE,IMAGE_SIZE)
    torch.onnx.export(model,dummy,str(onnx_path),input_names=['input'],output_names=['logits'],dynamic_axes={'input':{0:'batch'},'logits':{0:'batch'}},opset_version=17,dynamo=False,external_data=True)
    import onnxruntime as ort
    x,_=next(iter(test_dl)); x=x[:1]
    with torch.inference_mode(): ref=model(x).numpy()
    sess=ort.InferenceSession(str(onnx_path),providers=['CPUExecutionProvider'])
    actual=sess.run(['logits'],{'input':x.numpy()})[0]
    result['onnx_max_abs_diff']=float(np.max(np.abs(ref-actual)))
    result['onnx_argmax_matches']=bool(ref.argmax(1)[0]==actual.argmax(1)[0])
    (out_dir/'test_metrics.json').write_text(json.dumps(result,indent=2))
    print(json.dumps(result,indent=2))
    print('Artifacts:',out_dir)
    return result


## 4) เทรน Food-101

ถ้า Colab runtime หลุด ให้เชื่อม GPU ใหม่ รัน cells เตรียมข้อมูลและ function ซ้ำ แล้วรัน cell นี้อีกครั้ง ระบบจะ resume จาก `last_checkpoint.pt` โดยอัตโนมัติ

ถ้าต้องการเริ่มเทรนใหม่ตั้งแต่ต้นหลังจากมี checkpoint แล้ว ให้เปลี่ยนชื่อโฟลเดอร์ย่อยใน `food_out`/`thai_out` ก่อนรัน เพื่อเก็บผลเดิมไว้และไม่ resume checkpoint เก่า


In [ ]:
food_out = OUTPUT_ROOT / 'food101_baseline_retrain'
food_model, food_history, food_test_dl = train_model(
    name='Food-101', classes=food_classes, split=food_split,
    selection_metric='top1_accuracy', out_dir=food_out
)
food_metrics = final_evaluate_and_export('Food-101','food101',food_model,food_classes,food_test_dl,food_out)


In [ ]:
# เปรียบเทียบกับผลเดิมที่ใช้ split เดียวกัน (ตัวเลขอ้างอิงจากการรันเดิม)
old_food101 = {'top1_accuracy': 0.8501980198019802, 'top3_accuracy': 0.9384158415841584, 'macro_f1': 0.8500070163819173}
print('ตัวเดิม:', {k: round(v*100, 2) for k,v in old_food101.items()})
print('ตัวใหม่:', {k: round(food_metrics[k]*100, 2) for k in old_food101})
print('ผลต่าง (percentage points):', {k: round((food_metrics[k]-old_food101[k])*100, 2) for k in old_food101})


## 5) เทรน THFOOD-50

ชุดไทยมีจำนวนรูปต่อคลาสไม่เท่ากัน จึงเลือก checkpoint ด้วย validation Macro F1 และรายงานทั้ง Top-1, Top-3, Macro F1, Balanced Accuracy เพื่อไม่ให้คลาสที่มีภาพมากครอบงำภาพรวม


In [ ]:
thai_out = OUTPUT_ROOT / 'thfood50'
thai_model, thai_history, thai_test_dl = train_model(
    name='THFOOD-50', classes=thai_classes, split=thai_split,
    selection_metric='macro_f1', out_dir=thai_out
)
thai_metrics = final_evaluate_and_export('THFOOD-50','thfood50',thai_model,thai_classes,thai_test_dl,thai_out)


## 6) สรุปและไฟล์ที่ต้องส่งต่อ

โฟลเดอร์ `MyDrive/FoodCal_Retraining_2026/food101_baseline_retrain/` และ `.../thfood50/` จะมี ONNX, classes JSON/TXT, test metrics, per-class report, confusion matrix, best/last checkpoint และประวัติการเทรน

สำหรับหน้าเว็บ ให้ยืนยันก่อนว่าไฟล์ ONNX และ class list ของโมเดลไทยตรงกัน ห้ามใช้ confidence ระหว่างสองโมเดลเทียบกันโดยตรง และอย่าใส่โมเดล THFOOD-50 เข้า public demo จนกว่าจะตรวจ license/การนำไฟล์ dataset หรือโมเดลไปเผยแพร่ตามข้อกำหนดแล้ว
